# LLM 은 기억하지 못한다

ChatGPT 와 대화하면 앞 내용을 기억하는 것처럼 보입니다. **사실은 기억하지 않습니다.**

이 노트북에서 그것을 직접 확인하고, 어떻게 기억하는 것처럼 만드는지 배웁니다.

## 핵심 한 문장

> **모델은 매 호출마다 백지에서 시작합니다.**
> 우리가 보낸 `messages` 목록이 모델이 아는 전부입니다.

## 흐름

| 단계 | 내용 |
|---|---|
| Step 1~3 | 이름을 알려 주고 → 다시 물어보면 **모른다** |
| Step 4~5 | 지난 대화를 같이 보내면 → **기억한다** |
| Step 6~8 | 기억 없는 챗봇을 만들어 직접 겪어 보기 |

## Step 1. 준비

`openai` 와 함께 **`gradio`** 를 가져옵니다.
Step 8 에서 웹 채팅 화면을 몇 줄로 만들 때 씁니다.

> `IProgress not found` 경고가 떠도 무시해도 됩니다. 동작에 지장 없습니다.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

import os

load_dotenv(override=True)
# OpenAI 크레딧이 없어서 Gemini 로 보냅니다. (OpenAI 호환 주소 사용 - 1_2_gemini 참고)
# 크레딧을 충전하면 openai = OpenAI() 로 되돌리고 MODEL 을 gpt-5.4-nano 등으로 바꾸세요.
openai = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=os.getenv("GEMINI_URL"))
MODEL = "gemini-3.1-flash-lite"  # gemini-3.8-flash 는 무료 한도가 하루 20회라 lite 사용

d:\ai_edu_main\llm_edu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2. 이름을 알려 준다

```python
messages = [
    {"role": "system", "content": "당신은 IT 전문가입니다."},
    {"role": "user", "content": "나의 이름은 이상화입니다."}
]
```

모델에게 이름을 알려 주는 첫 번째 대화입니다.

In [2]:
messages = [
    {"role": "system", "content": "당신은 IT 전문가입니다."},
    {"role": "user", "content": "나의 이름은 이상화입니다."}
]

## Step 3. 모델이 이름을 불러 준다

```
안녕하세요, 이상화님! 😊
```

이름을 제대로 받았습니다. **우리가 보낸 메시지 안에 있었기 때문**입니다.

In [3]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

안녕하세요, 이상화님! 😊  
IT 관련해서 어떤 도움이 필요하신가요? (예: 문제 해결, 개발/서버 설정, 보안, 클라우드, 네트워크, 코딩 가이드 등)


## Step 4. 새 목록으로 이름을 물어본다

```python
messages = [
    {"role": "system", "content": "당신은 IT 전문가입니다"},
    {"role": "user", "content": "내 이름은 무엇인가요?"}
]
```

**`messages` 를 새로 만들었습니다.** Step 2 의 "이상화" 메시지는 여기에 없습니다.
사람이라면 방금 들었으니 기억하겠지만, 모델은 어떨까요?

In [4]:
messages = [
    {"role": "system", "content": "당신은 IT 전문가입니다"},
    {"role": "user", "content": "내 이름은 무엇인가요?"}
]

## Step 5. 모른다고 답한다 — 이 노트북의 핵심

```
지금 대화 내용만으로는 사용자님의 이름을 알 수 없습니다.
```

**불과 몇 초 전에 알려 준 이름을 모릅니다.**

고장이 아닙니다. 설계가 그렇습니다.

```
Step 3 호출  :  [system, user("이상화입니다")]        -> 답하고 끝. 아무것도 남기지 않는다
Step 5 호출  :  [system, user("내 이름은?")]          -> 완전히 새로운 요청
```

서버에는 **이전 호출의 흔적이 전혀 남지 않습니다.**
이것을 **상태 없음(stateless)** 이라고 합니다.

> ChatGPT 가 기억하는 것처럼 보이는 이유는
> **화면 뒤에서 지난 대화를 매번 통째로 다시 보내기 때문**입니다.

In [5]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

지금 대화 내용만으로는 사용자님의 이름을 알 수 없습니다.  
이름(또는 닉네임)을 알려주시면 그 이름으로 불러드릴게요.


## Step 6. 지난 대화를 같이 넣어 본다

```python
messages = [
    {"role": "system",    "content": "당신은 IT 전문가입니다."},
    {"role": "user",      "content": "나의 이름은 이상화입니다."},
    {"role": "assistant", "content": "반갑습니다, 이상화님! 무엇을 도와드릴까요?"},
    {"role": "user",      "content": "내 이름은 무엇인가요?"}
]
```

**`assistant` 역할**이 처음 등장했습니다. 모델이 **지난번에 한 답**을 우리가 적어 넣은 것입니다.

| role | 역할 |
|---|---|
| `system` | 규칙 |
| `user` | 사람이 한 말 |
| `assistant` | **모델이 한 말** — 대화를 이어 붙일 때 쓴다 |

이렇게 `user` → `assistant` → `user` 로 번갈아 쌓으면 대화 기록이 됩니다.

In [6]:
messages = [
    {"role": "system", "content": "당신은 IT 전문가입니다."},
    {"role": "user", "content": "나의 이름은 이상화입니다."},
    {"role": "assistant", "content": "반갑습니다, 이상화님! 무엇을 도와드릴까요?"},
    {"role": "user", "content": "내 이름은 무엇인가요?"}
]

## Step 7. 이제 기억한다

```
이상화님입니다.
```

같은 모델, 같은 질문인데 답이 달라졌습니다.
**모델이 똑똑해진 게 아니라 우리가 정보를 같이 보낸 것**뿐입니다.

### 그래서 기억은 누가 만드나

```
모델        매번 백지에서 시작한다
우리(개발자)  지난 대화를 보관했다가 매번 다시 보낸다   <-- 기억은 여기서 생긴다
```

이 일을 대신 해 주는 것이 LangChain 의 `memory`, OpenAI Agents SDK 의 `Session` 같은 장치입니다.
**안에서 하는 일은 지금 우리가 손으로 한 것과 같습니다.**

> 대화가 길어지면 매번 보내는 양도 계속 늘어납니다.
> 토큰 한도와 요금 문제가 생기는데, 이 이야기는 `1_7_context_openai` 에서 다룹니다.

In [7]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

이상화님입니다.


## Step 8-1. 챗봇용 `system` 프롬프트

이제 웹 채팅 화면을 만들어 **기억이 없다는 것을 몸으로 겪어** 봅니다.
상담사 역할을 주어 여러 번 주고받게 만듭니다.

In [8]:
system_prompt = """
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.
"""

## Step 8-2. 일부러 기억을 버리는 함수

```python
def chat_without_history(message, history):
    messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": message}]
```

함수 이름 그대로 **history 를 쓰지 않습니다.**

Gradio 는 지난 대화를 `history` 인자로 **친절하게 넘겨 줍니다.**
그런데 이 함수는 받기만 하고 **쓰지 않습니다.** 그래서 매번 백지에서 시작합니다.

> 일부러 이렇게 만들었습니다. 기억이 없으면 대화가 얼마나 답답한지 직접 느껴 보라는 것입니다.

### 고치려면

```python
def chat_with_history(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    #                                                           ^^^^^^^ 이것만 더하면 된다
```

`history` 를 목록에 끼워 넣기만 하면 됩니다.

In [9]:
def chat_without_history(message, history):
    messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

## Step 8-3. 채팅 화면 띄우기

```python
gr.ChatInterface(fn=chat_without_history, title="Without history")
```

`gr.ChatInterface` 는 함수 하나만 주면 **채팅 UI 를 통째로** 만들어 줍니다.
입력창·대화 목록·전송 버튼이 자동으로 생깁니다.

### 이렇게 시험해 보세요

```
나   : 내 이름은 홍길동이야
봇   : 반갑습니다 홍길동님
나   : 내 이름이 뭐라고 했지?
봇   : ...                     <-- 모릅니다
```

화면에는 대화가 쭉 보이는데도 모델은 모릅니다.
**화면에 보이는 것과 모델이 받는 것은 다릅니다.**

> 끝나면 `chat_ui.close()` 로 서버를 내리거나 셀을 중지하세요.
> 그냥 두면 포트 7860 을 계속 잡고 있어 다음 실행에서 충돌합니다.

In [10]:
dark_css = """
.gradio-container, .gradio-container .fillable, body {
    background-color: #000000 !important;
}
"""

with gr.Blocks() as chat_ui:
    gr.ChatInterface(fn=chat_without_history, title="Without history")

chat_ui.launch(css=dark_css, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


---

## 정리

| | 내용 |
|---|---|
| **모델은 백지에서 시작** | 이전 호출의 흔적이 서버에 없다 |
| **`messages` 가 전부** | 보낸 것만 안다 |
| **`assistant` 로 쌓는다** | 모델의 지난 답을 넣어야 대화가 이어진다 |
| **기억은 우리가 만든다** | 프레임워크의 memory 도 안에서는 같은 일을 한다 |

### 직접 해 보기

1. `chat_without_history` 를 `history` 쓰도록 고치고 다시 띄워 보세요.
2. Step 6 에서 `assistant` 줄만 빼고 돌려 보세요. 다시 모른다고 하나요?
3. `assistant` 내용을 **거짓("반갑습니다, 김철수님!")** 으로 바꾸면 모델은 누구라고 답할까요?
   (모델은 받은 것을 사실로 믿습니다. 이것이 프롬프트 주입 공격의 출발점입니다.)

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_6_2_stateful` | 기억을 제대로 붙인 챗봇 |
| `1_7_context_openai` | 대화가 길어질 때 토큰 한도를 다루는 방법 |